# Content Analysis Pipeline — BigQuery AI Functions

An end-to-end content analysis pipeline that composes four AI functions together:

1. **Generate** sample product reviews with `AI.GENERATE_TABLE`
2. **Classify** each review by topic with `AI.CLASSIFY`
3. **Score** each review for urgency with `AI.SCORE`
4. **Summarize** findings with `AI.GENERATE`

**What this demonstrates:**
- Composing multiple AI functions in a single analytical pipeline
- Using `AI.GENERATE_TABLE` to create realistic sample data
- Combining managed functions (`AI.CLASSIFY`, `AI.SCORE`) with generation functions
- Aggregating AI-enriched data into executive summaries

**Functions used:** [`AI.GENERATE_TABLE`](../../functions/ai_generate_table/) | [`AI.CLASSIFY`](../../functions/ai_classify/) | [`AI.SCORE`](../../functions/ai_score/) | [`AI.GENERATE`](../../functions/ai_generate/)

**Prerequisites:** [Setup guide](../../setup/) | [Function reference](../../RESOURCES.md)

---
## Setup

Set your project and location, authenticate, and create shared resources.

> This workflow requires a connection and a remote model for `AI.GENERATE_TABLE`. See the [Setup Reference](../../setup/) for details.

In [12]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks
CONNECTION_ID = 'bq_ai_functions'  # Shared connection

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [13]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm')

In [14]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [15]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready
The bigquery_magics extension is already loaded. To reload it, use:
  %reload_ext bigquery_magics


In [16]:
import subprocess as _sp, json as _json

# Create connection (idempotent)
_sp.run(['bq', 'mk', '--connection', '--location', LOCATION,
         '--connection_type', 'CLOUD_RESOURCE',
         '--project_id', PROJECT_ID, CONNECTION_ID],
        capture_output=True, text=True)

# Get service account and grant Vertex AI User role
r = _sp.run(['bq', 'show', '--connection', '--format=json',
             '--project_id', PROJECT_ID, '--location', LOCATION, CONNECTION_ID],
            capture_output=True, text=True, check=True)
sa = _json.loads(r.stdout)['cloudResource']['serviceAccountId']
_sp.run(['gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
         f'--member=serviceAccount:{sa}', '--role=roles/aiplatform.user', '--quiet'],
        capture_output=True, text=True)
print(f'Connection {CONNECTION_ID} ready (SA: {sa})')

Connection bq_ai_functions ready (SA: bqcx-1026793852137-hx0g@gcp-sa-bigquery-condel.iam.gserviceaccount.com)


In [17]:
# Create remote Gemini model (idempotent)
client.query(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`
  REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
  OPTIONS (endpoint = 'gemini-2.5-flash')
''').result()
print('Model gemini_flash ready')

Model gemini_flash ready


---
## Step 1 — Generate sample reviews with AI.GENERATE_TABLE

Use `AI.GENERATE_TABLE` to create realistic product reviews. Each input row gets one output row — so we provide multiple product categories to generate a diverse set of reviews.

In [18]:
output_schema = """review_text STRING OPTIONS(description = "The full review text written by the customer"),
       product_name STRING OPTIONS(description = "Specific product name mentioned or inferred"),
       star_rating INT64 OPTIONS(description = "Star rating from 1 to 5")"""

query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_content_reviews` AS
SELECT review_text, product_name, star_rating
FROM AI.GENERATE_TABLE(
  MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`,
  (SELECT CONCAT(
     'Write a realistic customer review for a ', product_type,
     ' product. Make some reviews very positive, some negative, and some mixed. ',
     'Vary the length and writing style.'
   ) AS prompt
   FROM UNNEST([
     'wireless headphones', 'laptop stand', 'USB-C hub', 'mechanical keyboard',
     'portable monitor', 'webcam', 'desk lamp', 'ergonomic mouse',
     'noise machine', 'cable management kit', 'monitor arm', 'standing desk mat'
   ]) AS product_type),
  STRUCT(
    """{output_schema}""" AS output_schema
  )
)
'''
client.query(query).result()

reviews = client.query(
    f'SELECT product_name, star_rating, LEFT(review_text, 80) AS review_preview FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_reviews`'
).to_dataframe()
print(f'{len(reviews)} reviews generated')
reviews

12 reviews generated


,product_name,star_rating,review_preview
0,ProHub 8-in-1 USB-C Dock,5,Absolutely thrilled with this USB-C hub! I needed something to expand the ports
1,Keychron K2 V2,5,"I've been using this keyboard for about a month now, and I can honestly say it's"
2,Ergotron LX Desk Monitor Arm,5,"Seriously, if you're on the fence about getting a monitor arm, just do it. And i"
3,FlexiRise Laptop Stand,5,"I absolutely love this laptop stand! I've been working from home for months, and"
4,ErgoComfort Standing Mat,5,I can't believe I waited this long to get a standing desk mat! This ErgoComfort
5,Luminaire Flexi-Desk Lamp,5,I absolutely LOVE this desk lamp! It's exactly what I needed for my home office
6,ErgoGlide Vertical Mouse,5,I've been suffering from wrist pain for months due to extensive computer use. My
7,Premium Desk Cable Management Kit,5,I can't believe I lived without this for so long! My desk was a tangled mess of
8,ZenScreen Go 15.6,5,Absolutely love this monitor! I travel a lot for work and this has been a game-c
9,SoundFlow Xtreme Wireless Earbuds,5,I can't say enough good things about these earbuds! The sound quality is just ph


---
## Step 2 — Classify reviews by topic with AI.CLASSIFY

Use `AI.CLASSIFY` to categorize each review into a topic. BigQuery auto-optimizes the classification prompt.

In [19]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_content_classified` AS
SELECT
  review_text,
  product_name,
  star_rating,
  AI.CLASSIFY(
    review_text,
    [('quality', 'About product build quality, durability, or materials'),
     ('usability', 'About ease of use, setup, or user experience'),
     ('value', 'About price, value for money, or cost-effectiveness'),
     ('support', 'About customer service, warranty, or returns'),
     ('features', 'About specific product features or capabilities')]
  ) AS topic
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_reviews`
'''
client.query(query).result()

# Show analytical insight: topic × star rating cross-tabulation
classified = client.query(
    f'SELECT topic, star_rating, product_name FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_classified`'
).to_dataframe()
print('Topic distribution by star rating:')
cross_tab = classified.groupby('topic')['star_rating'].agg(['count', 'mean']).round(1)
cross_tab.columns = ['reviews', 'avg_stars']
print(cross_tab.sort_values('reviews', ascending=False).to_string())
print(f'\nProducts by topic:')
for topic in classified['topic'].unique():
    products = classified[classified['topic'] == topic]['product_name'].tolist()
    print(f'  {topic}: {", ".join(products)}')

Topic distribution by star rating:
           reviews  avg_stars
topic                        
features         6        4.7
usability        6        5.0

Products by topic:
  features: LogiTech HD Pro C920, SoundFlow Xtreme Wireless Earbuds, Dream Weaver Sound Machine, Keychron K2 V2, ProHub 8-in-1 USB-C Dock, Luminaire Flexi-Desk Lamp
  usability: Ergotron LX Desk Monitor Arm, FlexiRise Laptop Stand, ErgoComfort Standing Mat, ErgoGlide Vertical Mouse, ZenScreen Go 15.6, Premium Desk Cable Management Kit


---
## Step 3 — Score reviews for urgency with AI.SCORE

Use `AI.SCORE` to rate each review's urgency on a 1–10 scale. This helps prioritize which reviews need immediate attention — surfacing product issues that require action.

In [20]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_content_scored` AS
SELECT
  review_text,
  product_name,
  star_rating,
  topic,
  AI.SCORE(CONCAT(
    'Rate the urgency of this customer review on a scale of 1 to 10, ',
    'where 1 is no action needed and 10 is requires immediate attention ',
    '(e.g. safety issue, defective product, very unhappy customer): ',
    review_text
  )) AS urgency
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_classified`
'''
client.query(query).result()

scored = client.query(
    f'SELECT product_name, star_rating, topic, urgency FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_scored` ORDER BY urgency DESC'
).to_dataframe()

# Urgency distribution
print('Urgency breakdown:')
scored['urgency_level'] = scored['urgency'].apply(
    lambda x: 'HIGH (7-10)' if x >= 7 else ('MEDIUM (4-6)' if x >= 4 else 'LOW (1-3)')
)
print(scored['urgency_level'].value_counts().to_string())
print(f'\nAll reviews ranked by urgency:')
scored[['product_name', 'star_rating', 'topic', 'urgency']]

Urgency breakdown:
urgency_level
LOW (1-3)       11
MEDIUM (4-6)     1

All reviews ranked by urgency:


,product_name,star_rating,topic,urgency
0,LogiTech HD Pro C920,3,features,6.0
1,Keychron K2 V2,5,features,3.0
2,ProHub 8-in-1 USB-C Dock,5,features,1.0
3,Dream Weaver Sound Machine,5,features,1.0
4,Luminaire Flexi-Desk Lamp,5,features,1.0
5,SoundFlow Xtreme Wireless Earbuds,5,features,1.0
6,ErgoComfort Standing Mat,5,usability,1.0
7,FlexiRise Laptop Stand,5,usability,1.0
8,Premium Desk Cable Management Kit,5,usability,1.0
9,Ergotron LX Desk Monitor Arm,5,usability,1.0


### Action items — high urgency reviews

These reviews scored 5+ urgency and may need immediate attention from product teams.

In [21]:
action_items = client.query(f'''
  SELECT product_name, star_rating, topic, urgency, review_text
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_scored`
  WHERE urgency >= 5
  ORDER BY urgency DESC
''').to_dataframe()

if len(action_items) == 0:
    print('No high-urgency reviews — all products are in good shape!')
else:
    print(f'{len(action_items)} reviews need attention:\n')
    for _, row in action_items.iterrows():
        print(f'[Urgency {row["urgency"]:.0f}] {row["product_name"]} ({row["topic"]}, {row["star_rating"]} stars)')
        print(f'  {row["review_text"][:200]}')
        print()

1 reviews need attention:

[Urgency 6] LogiTech HD Pro C920 (features, 3 stars)
  I bought this LogiTech HD Pro C920 a few weeks ago for work meetings. The video quality is definitely a step up from my laptop's built-in camera, especially in good lighting. Colors are vibrant and it



---
## Step 4 — Summarize findings with AI.GENERATE

Aggregate all the classified and scored reviews, then use `AI.GENERATE` to produce an executive summary of the findings.

In [22]:
query = f'''
WITH summary_data AS (
  SELECT
    COUNT(*) AS total_reviews,
    ROUND(AVG(star_rating), 1) AS avg_stars,
    ROUND(AVG(urgency), 1) AS avg_urgency,
    COUNTIF(urgency >= 7) AS high_urgency_count,
    STRING_AGG(
      CONCAT(product_name, ' (', topic, ', ', star_rating, ' stars, urgency ', CAST(urgency AS STRING), '): ', review_text),
      ' | '
    ) AS all_reviews
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_content_scored`
)
SELECT (AI.GENERATE(
  CONCAT(
    'You are a product analytics manager. Analyze these customer reviews and provide an executive summary. ',
    'Include: (1) overall sentiment overview, (2) top issues by topic, (3) urgent items needing immediate action, ',
    '(4) recommendations for product teams. ',
    'Stats: ', CAST(s.total_reviews AS STRING), ' reviews, ',
    'avg rating ', CAST(s.avg_stars AS STRING), '/5, ',
    'avg urgency ', CAST(s.avg_urgency AS STRING), '/10, ',
    CAST(s.high_urgency_count AS STRING), ' high-urgency reviews. ',
    'Reviews: ', s.all_reviews
  )
)).result AS executive_summary
FROM summary_data s
'''
df = client.query(query).to_dataframe()
print(df.iloc[0]['executive_summary'])

**Executive Summary: Customer Review Analysis - Q4 2023**

**To:** Product Leadership Team
**From:** Product Analytics Manager
**Date:** October 26, 2023
**Subject:** Analysis of Recent Customer Reviews

This report summarizes insights from 12 recent customer reviews, focusing on sentiment, key issues, and recommendations for product development.

---

**(1) Overall Sentiment Overview**

Customer sentiment is overwhelmingly positive, with an impressive **average rating of 4.8 out of 5 stars** across all products reviewed. The average urgency score, indicating the severity of reported issues, is very low at **1.6 out of 10**, with **no high-urgency reviews (defined as 7+)** identified in this dataset.

Eleven out of twelve products received a perfect 5-star rating, highlighting strong customer satisfaction with product features, usability, and problem-solving capabilities. The sole exception was the LogiTech HD Pro C920 webcam, which received 3 stars due to specific functional shortcomi

---
## Cleanup

Drop tables created by this workflow. Shared resources (dataset, models, connection) are left for other notebooks.

In [ ]:
for table in ['workflow_content_reviews', 'workflow_content_classified', 'workflow_content_scored']:
    client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.{table}', not_found_ok=True)
    print(f'Table {table} deleted')


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')